<a href="https://colab.research.google.com/github/MudadlaPranay12/Pranay_GenAI_MAY_2026/blob/main/resumeparser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 70.2 MB/s eta 0:00:00


In [4]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.1 MB/s eta 0:00:00


In [17]:

import json
import re
import google.generativeai as genai
import typing_extensions as typing

from google.colab import userdata
from PyPDF2 import PdfReader

genai.configure(
    api_key=userdata.get("GEMINI_API_KEYS")
)

model = genai.GenerativeModel(

    "gemini-2.5-flash-lite",

    system_instruction="""
You are an advanced resume parser.

RULES:

1. Extract only information explicitly present in the resume.

2. Never invent or hallucinate information.

3. If a field is missing, return null.

4. Output must always be valid JSON only.

5. Normalize phone numbers to +91XXXXXXXXXX whenever possible.

6. Skills must be clean technology names without duplicates.

7. Summary must be exactly one sentence.

8. Resume may be written in English or any Indian language.

9. Output values must always be in English.

10. Do not add extra keys outside the required schema.
"""
)


class Education(typing.TypedDict):

    degree: str | None
    institute: str | None
    year: str | None
    score: str | None


class Experience(typing.TypedDict):

    company: str | None
    role: str | None
    duration: str | None
    highlights: list[str]


class Project(typing.TypedDict):

    name: str | None
    description: str | None
    tech_stack: list[str]


class Resume(typing.TypedDict):

    name: str | None
    email: str | None
    phone: str | None
    linkedin_url: str | None

    education: list[Education]
    skills: list[str]
    experience: list[Experience]
    projects: list[Project]

    total_experience_years: float
    summary: str | None

# =========================================================
# EXPECTED KEYS
# =========================================================

EXPECTED_KEYS = [
    "name",
    "email",
    "phone",
    "linkedin_url",
    "education",
    "skills",
    "experience",
    "projects",
    "total_experience_years",
    "summary"
]

# =========================================================
# SAFE JSON PARSER
# =========================================================

def safe_json_parse(raw_text):

    try:

        raw_text = raw_text.strip()

        raw_text = raw_text.replace("```json", "")
        raw_text = raw_text.replace("```", "")

        json_start = raw_text.find("{")
        json_end = raw_text.rfind("}")

        if json_start != -1 and json_end != -1:

            raw_text = raw_text[json_start:json_end + 1]

        return json.loads(raw_text)

    except Exception as e:

        print("JSON Parsing Error:", e)

        return None

# =========================================================
# VALIDATION FUNCTION
# =========================================================

def validate_data(data: dict) -> dict:

    if data is None:

        data = {}

    # =====================================================
    # ENSURE EXACT 10 KEYS
    # =====================================================

    cleaned = {}

    for key in EXPECTED_KEYS:

        cleaned[key] = data.get(key, None)

    data = cleaned

    # =====================================================
    # EMAIL VALIDATION
    # =====================================================

    email = data.get("email")

    if email:

        if "@" not in email:

            data["email"] = None

    # =====================================================
    # PHONE NORMALIZATION
    # =====================================================

    phone = data.get("phone")

    if phone:

        digits = re.sub(r"\D", "", str(phone))

        if len(digits) >= 10:

            digits = digits[-10:]

            data["phone"] = "+91" + digits

        else:

            data["phone"] = None

    # =====================================================
    # LINKEDIN URL NORMALIZATION
    # =====================================================

    linkedin = data.get("linkedin_url")

    if linkedin:

        if not linkedin.startswith("http"):

            data["linkedin_url"] = "https://" + linkedin

    # =====================================================
    # EXPERIENCE VALIDATION
    # =====================================================

    try:

        exp = float(data.get("total_experience_years"))

        if exp < 0:

            data["total_experience_years"] = 0.0

    except:

        data["total_experience_years"] = 0.0

    # =====================================================
    # EDUCATION YEAR VALIDATION
    # =====================================================

    education = data.get("education")

    if education:

        for edu in education:

            year = edu.get("year")

            if year:

                numbers = re.findall(r"\d{4}", str(year))

                if numbers:

                    y = int(numbers[0])

                    if y < 1990 or y > 2026:

                        edu["year"] = None

    # =====================================================
    # REMOVE DUPLICATE SKILLS
    # =====================================================

    skills = data.get("skills")

    if skills:

        unique_skills = []

        for skill in skills:

            skill = skill.strip()

            if skill not in unique_skills:

                unique_skills.append(skill)

        data["skills"] = unique_skills

    return data

# =========================================================
# MAIN PARSER FUNCTION
# =========================================================

def parse_resume(text: str) -> dict:

    prompt = f"""
Extract resume details from the following text.

Return ONLY valid JSON.

Required keys:
name
email
phone
linkedin_url
education
skills
experience
projects
total_experience_years
summary

Resume Text:
{text}
"""

    # =====================================================
    # WAY 1 — PROMPT ONLY
    # =====================================================

    print("\n================ WAY 1 — PROMPT ONLY ================\n")

    response_1 = model.generate_content(prompt)

    print(response_1.text)

    try:

        json.loads(response_1.text)

        print("\nWAY 1 JSON Parsing: SUCCESS")

    except:

        print("\nWAY 1 JSON Parsing: FAILED")

    # =====================================================
    # WAY 2 — MIME TYPE
    # =====================================================

    print("\n================ WAY 2 — MIME TYPE ================\n")

    data_2 = None

    for attempt in range(2):

        try:

            response_2 = model.generate_content(

                prompt,

                generation_config={

                    "response_mime_type": "application/json",
                    "temperature": 0.2,
                    "max_output_tokens": 2048
                }
            )

            data_2 = safe_json_parse(response_2.text)

            if data_2 is not None:

                print("WAY 2 Parsing Success\n")

                break

        except Exception as e:

            print(f"WAY 2 Attempt {attempt + 1} Failed")
            print("Error:", e)

    if data_2 is None:

        data_2 = {

            "name": None,
            "email": None,
            "phone": None,
            "linkedin_url": None,
            "education": [],
            "skills": [],
            "experience": [],
            "projects": [],
            "total_experience_years": 0.0,
            "summary": None
        }

    data_2 = validate_data(data_2)

    print(json.dumps(data_2, indent=2, ensure_ascii=False))

    # =====================================================
    # WAY 3 — SCHEMA
    # =====================================================

    print("\n================ WAY 3 — SCHEMA ================\n")

    data_3 = None

    for attempt in range(2):

        try:

            response_3 = model.generate_content(

                prompt,

                generation_config={

                    "response_mime_type": "application/json",
                    "response_schema": Resume,
                    "temperature": 0.2,
                    "max_output_tokens": 2048
                }
            )

            data_3 = safe_json_parse(response_3.text)

            if data_3 is not None:

                print("WAY 3 Parsing Success\n")

                break

        except Exception as e:

            print(f"WAY 3 Attempt {attempt + 1} Failed")
            print("Error:", e)

    if data_3 is None:

        print("WAY 3 completely failed.")

        data_3 = {

            "name": None,
            "email": None,
            "phone": None,
            "linkedin_url": None,
            "education": [],
            "skills": [],
            "experience": [],
            "projects": [],
            "total_experience_years": 0.0,
            "summary": None
        }

    data_3 = validate_data(data_3)

    print(json.dumps(data_3, indent=2, ensure_ascii=False))

    # =====================================================
    # COMPARISON
    # =====================================================

    print("\n================ COMPARISON ================\n")

    print("""
1. Prompt-only method may return invalid JSON or extra text.

2. MIME type produces cleaner JSON responses.

3. Schema-based output is the most structured and reliable.

4. Schema reduces hallucinated fields and formatting issues.
""")

    # =====================================================
    # SUMMARY
    # =====================================================

    print("\n================ SUMMARY ================\n")

    print("Name:", data_3.get("name"))
    print("Email:", data_3.get("email"))
    print("Phone:", data_3.get("phone"))
    print("Top Skills:", data_3.get("skills"))
    print("Total Experience:", data_3.get("total_experience_years"))

    # =====================================================
    # RETURN FINAL OUTPUT
    # =====================================================

    return data_3

# =========================================================
# TEST 1 — WELL FORMATTED RESUME
# =========================================================

resume_1 = """
ANJALI SHARMA

Email: anjali.sharma@gmail.com | Phone: 9876543210

LinkedIn: linkedin.com/in/anjali-sharma

EDUCATION

- B.Tech in Computer Science, IIT Delhi, 2024 (CGPA: 8.9)

- 12th CBSE, DPS Noida, 2020 (94%)

SKILLS

Python, Django, FastAPI, React.js, MongoDB, PostgreSQL,
Docker, Git, AWS (basics), TensorFlow

EXPERIENCE

Software Engineering Intern, Flipkart (May 2023 - July 2023)

- Built a recommendation microservice using Python + Redis

- Reduced API latency by 35%

Open Source Contributor, scikit-learn (2022 - present)

- Merged 4 PRs related to model serialization

PROJECTS

- AgriBot: WhatsApp chatbot for farmers using Gemini API + Twilio

- StudyMate: Note-summarizer Chrome extension (1.2k users)
"""

print("\n================ TEST 1 ================\n")

parsed_1 = parse_resume(resume_1)

# =========================================================
# TEST 2 — MESSY RESUME
# =========================================================

resume_2 = """
hii i am ravi (ravi.k@yahoo.in / 8765-432-100). recently finished
my btech from vit vellore (2023, cgpa around 8). i know java/python/c++
and a bit of ml stuff. did an internship at zoho for 6 months as software
trainee. also worked at a startup called bytemate for 8 months as backend
developer where i built their payment gateway. love football and travelling.
also made a small project called NoteShare - a notes sharing app for
college students built with react and firebase.
"""

print("\n================ TEST 2 ================\n")

parsed_2 = parse_resume(resume_2)

# =========================================================
# TEST 3 — PDF RESUME
# =========================================================

print("\n================ TEST 3 — PDF ================\n")

pdf_path = "/content/sample_resume.pdf"

try:

    reader = PdfReader(pdf_path)

    pdf_text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:

            pdf_text += extracted

    parsed_3 = parse_resume(pdf_text)

except Exception as e:

    print("PDF processing failed.")
    print("Error:", e)


================ TEST 1 ================


================ WAY 1 — PROMPT ONLY ================

```json
{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin_url": "https://linkedin.com/in/anjali-sharma",
  "education": [
    {
      "degree": "B.Tech in Computer Science",
      "institution": "IIT Delhi",
      "year": 2024,
      "cgpa": 8.9
    },
    {
      "degree": "12th CBSE",
      "institution": "DPS Noida",
      "year": 2020,
      "percentage": 94.0
    }
  ],
  "skills": [
    "Python",
    "Django",
    "FastAPI",
    "React.js",
    "MongoDB",
    "PostgreSQL",
    "Docker",
    "Git",
    "AWS",
    "TensorFlow"
  ],
  "experience": [
    {
      "title": "Software Engineering Intern",
      "company": "Flipkart",
      "start_date": "May 2023",
      "end_date": "July 2023",
      "description": "Built a recommendation microservice using Python + Redis. Reduced API latency by 35%."
    },
    {
      "title": "Op